# 01 - Sources, datasets et instituts

Ce notebook sert de point d'entrée. Il ne trace pas de courbes électorales. Il répond d'abord à une question simple : qu'est-ce qu'on a réellement comme données, par dataset et par institut, et où sont les trous méthodologiques.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from presidentielle2027.extraction.canonicalization import canonicalize_candidate_fields, is_generic_bloc_label


def load_poll_frame() -> tuple[Path, pd.DataFrame]:
    candidates = [
        REPO_ROOT / "data" / "processed" / "wikipedia_2027_polls_normalized_v2.csv",
        REPO_ROOT / "data" / "processed" / "wikipedia_2027_polls_normalized.csv",
        REPO_ROOT / "data" / "processed" / "sample_polls.csv",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate, pd.read_csv(candidate)
    raise FileNotFoundError("Aucun CSV exploitable trouvé dans data/processed.")


dataset_path, frame = load_poll_frame()
dataset_registry = pd.read_csv(REPO_ROOT / "data" / "reference" / "datasets_metadata.csv", engine="python", on_bad_lines="skip")
pollster_registry = pd.read_csv(REPO_ROOT / "data" / "reference" / "pollsters_metadata.csv", engine="python", on_bad_lines="skip")

canonical = frame.apply(
    lambda row: canonicalize_candidate_fields(
        row.get("candidate_name"),
        row.get("candidate_party"),
        row.get("political_family"),
    ),
    axis=1,
    result_type="expand",
)
canonical.columns = ["candidate_name", "candidate_party", "political_family"]
frame[["candidate_name", "candidate_party", "political_family"]] = canonical
frame["publication_date"] = pd.to_datetime(frame["publication_date"], errors="coerce")
frame["fieldwork_start_date"] = pd.to_datetime(frame.get("fieldwork_start_date"), errors="coerce")
frame["fieldwork_end_date"] = pd.to_datetime(frame.get("fieldwork_end_date"), errors="coerce")
frame["sample_size"] = pd.to_numeric(frame.get("sample_size"), errors="coerce")
frame["estimate_percent"] = pd.to_numeric(frame.get("estimate_percent"), errors="coerce")
frame["extraction_confidence"] = pd.to_numeric(frame.get("extraction_confidence"), errors="coerce")
frame["is_generic_bloc"] = frame["candidate_name"].map(is_generic_bloc_label)

pd.Series({
    "active_dataset": dataset_path.name,
    "rows": len(frame),
    "polls": frame["poll_id"].nunique() if "poll_id" in frame.columns else len(frame),
    "rounds": frame["round"].nunique(dropna=True),
    "scenarios": frame["scenario_name"].nunique(dropna=True),
    "pollsters": frame["polling_company"].nunique(dropna=True),
    "candidates": frame["candidate_name"].nunique(dropna=True),
})

In [ ]:
display(dataset_registry)

dataset_facts = (
    frame.groupby(["source_name", "round"], dropna=False)
    .agg(
        rows=("poll_id", "count"),
        scenarios=("scenario_name", "nunique"),
        pollsters=("polling_company", "nunique"),
        first_publication=("publication_date", "min"),
        last_publication=("publication_date", "max"),
        missing_sample_size=("sample_size", lambda s: int(s.isna().sum())),
        missing_collection_method=("collection_method", lambda s: int(s.isna().sum())),
        missing_commissioner=("commissioner", lambda s: int(s.isna().sum())),
    )
    .reset_index()
    .sort_values(["rows", "last_publication"], ascending=[False, False])
)
display(dataset_facts)

In [ ]:
display(pollster_registry)

pollster_facts = (
    frame.groupby("polling_company", dropna=False)
    .agg(
        rows=("poll_id", "count"),
        rounds=("round", "nunique"),
        scenarios=("scenario_name", "nunique"),
        candidates=("candidate_name", "nunique"),
        first_publication=("publication_date", "min"),
        last_publication=("publication_date", "max"),
        average_sample_size=("sample_size", "mean"),
        missing_commissioner=("commissioner", lambda s: int(s.isna().sum())),
        missing_media_partner=("media_partner", lambda s: int(s.isna().sum())),
    )
    .reset_index()
)
pollster_facts = pollster_facts.merge(pollster_registry, on="polling_company", how="left")
pollster_facts.sort_values(["rows", "last_publication"], ascending=[False, False])

In [ ]:
critical_columns = [
    "source_url",
    "polling_company",
    "commissioner",
    "media_partner",
    "fieldwork_start_date",
    "fieldwork_end_date",
    "publication_date",
    "sample_size",
    "population",
    "collection_method",
    "quota_method",
    "round",
    "scenario_name",
    "candidate_name",
    "candidate_party",
    "political_family",
    "estimate_percent",
    "extraction_confidence",
]

coverage = pd.DataFrame([
    {
        "column": column,
        "missing_count": int(frame[column].isna().sum()),
        "missing_rate": frame[column].isna().mean(),
        "distinct_values": frame[column].nunique(dropna=True),
    }
    for column in critical_columns
    if column in frame.columns
]).sort_values(["missing_rate", "column"], ascending=[False, True])
display(coverage)

problem_rows = frame.loc[
    frame[["polling_company", "scenario_name", "candidate_name", "publication_date", "estimate_percent"]].isna().any(axis=1)
    | frame["sample_size"].isna(),
    [
        "poll_id",
        "source_name",
        "polling_company",
        "round",
        "scenario_name",
        "candidate_name",
        "sample_size",
        "publication_date",
        "source_url",
    ],
]
problem_rows.head(40)